# Working with agents

In [ ]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.tools import tool

# Load GROQ_API_KEY from the local .env file.
load_dotenv()

inventory_model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
)


## A basic agent with one tool

In [ ]:
@tool
def check_stock_service(item: str) -> str:
    """Return the remaining stock quantity for an inventory item."""
    stock_by_item = {
        "headphone": 6,
        "keyboard": 11,
        "laptop": 12,
    }
    quantity = stock_by_item.get(item.lower())
    if quantity is None:
        return f"{item} is not present in the stock database."
    return f"{item} has {quantity} units left in stock."


basic_inventory_agent = create_agent(
    model=inventory_model,
    tools=[check_stock_service],
)

basic_agent_result = basic_inventory_agent.invoke(
    {
        "messages": [
            SystemMessage("You are a helpful inventory assistant."),
            HumanMessage("How many laptops do we have?"),
        ]
    }
)

print(basic_agent_result["messages"][-1].content)


## Give an agent more than one tool

In [ ]:
@tool
def get_price_service(item: str) -> str:
    """Return the price in rupees for an inventory item."""
    price_by_item = {
        "headphone": 1999,
        "keyboard": 1499,
        "laptop": 54999,
    }
    price = price_by_item.get(item.lower())
    if price is None:
        return f"{item} does not have a price in the catalogue."
    return f"{item} costs Rs {price}."


shopping_agent = create_agent(
    model=inventory_model,
    tools=[check_stock_service, get_price_service],
)

multi_tool_result = shopping_agent.invoke(
    {
        "messages": [
            SystemMessage("You are a helpful inventory assistant."),
            HumanMessage("Do we have laptops in stock, and what is the price?"),
        ]
    }
)

# Inspect the messages to see the agent's tool requests and final answer.
for agent_message in multi_tool_result["messages"]:
    print(f"{type(agent_message).__name__}: {agent_message.content}")
